# DeepLabV3+ Fine-Tuning — **4 Bands (R,G,B,NIR)** from Planet 8-Band (SuperDove SR)

This notebook trains a **DeepLabV3+** on **four bands** extracted from Planet **8‑band** GeoTIFF tiles (size **512×512**):
**R, G, B, NIR**. We use **ImageNet-pretrained** encoders and inflate the first conv 3→4 channels (NIR weights = mean of RGB).

## Dependencies

In [1]:

%pip install -q torch torchvision timm segmentation-models-pytorch rasterio albumentations pandas scikit-image


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 117.1 MB/s eta 0:00:00


## Configuration

In [10]:

from pathlib import Path
import random

IMAGES_DIR = Path("/content/drive/My Drive/MS Data Science/Thesis/Training/3 Preparing train data/8_band_outputs/subset_800/images")  # change to your imagery folder
LABELS_DIR = Path("/content/drive/My Drive/MS Data Science/Thesis/Training/3 Preparing train data/8_band_outputs/subset_800/labels")  # change to your label folder (same filenames)


# SuperDove SR band map (1-indexed): 1 Coastal, 2 Blue, 3 Green I, 4 Green, 5 Yellow, 6 Red, 7 Red Edge, 8 NIR
BANDS = (1, 2, 3, 4, 5, 6, 7, 8)  # (R,G,B,NIR)

OUTPUT_DIR = Path("/content/drive/My Drive/MS Data Science/Thesis/Training/4 Model Training/U-Net-plus-plus/training_output_finetune")
(OUTPUT_DIR / "logs").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "models").mkdir(parents=True, exist_ok=True)

NUM_SAMPLES = 800
RANDOM_SEED = 42
VAL_RATIO = 0.2
BATCH_SIZE = 2
NUM_WORKERS = 0
EPOCHS = 10
LR = 1e-3

BACKBONES = ["timm-efficientnet-b0"]

random.seed(RANDOM_SEED)


## Discover Pairs & Sample

In [5]:

def list_pairs(images_dir: Path, labels_dir: Path):
    img_files = {p.name: p for p in images_dir.glob("*.tif")}
    lbl_files = {p.name: p for p in labels_dir.glob("*.tif")}
    common = sorted(set(img_files) & set(lbl_files))
    return [(img_files[n], lbl_files[n]) for n in common]

all_pairs = list_pairs(IMAGES_DIR, LABELS_DIR)
if not all_pairs:
    raise RuntimeError(f"No paired .tif files found under {IMAGES_DIR} & {LABELS_DIR}.")

print(f"Found {len(all_pairs)} pairs.")
random.shuffle(all_pairs)
sampled_pairs = all_pairs[:min(NUM_SAMPLES, len(all_pairs))]

split_idx = int(len(sampled_pairs) * (1 - VAL_RATIO))
train_pairs = sampled_pairs[:split_idx]
val_pairs = sampled_pairs[split_idx:]
print(f"Train: {len(train_pairs)}, Val: {len(val_pairs)}")


Found 800 pairs.
Train: 640, Val: 160


## Dataset & Dataloaders

In [6]:

import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import rasterio
import albumentations as A

def read_geotiff(path: Path):
    with rasterio.open(path) as src:
        arr = src.read()  # (C,H,W)
    return arr

def select_rgbnir(x: np.ndarray, bands=(6,4,2,8)):
    idx = [b-1 for b in bands]
    return x[idx, ...].astype(np.float32)

def minmax_per_band(x: np.ndarray, eps=1e-6):
    x = x.astype(np.float32)
    for i in range(x.shape[0]):
        vmin = np.min(x[i]); vmax = np.max(x[i])
        if vmax - vmin < eps:
            x[i] = 0.0
        else:
            x[i] = (x[i] - vmin) / (vmax - vmin + eps)
    return x

class LakeTiles(Dataset):
    def __init__(self, pairs, bands=(1, 2, 3, 4, 5, 6, 7, 8), aug=None, normalize=True):
        self.pairs = pairs
        self.bands = bands
        self.aug = aug
        self.normalize = normalize

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, lbl_path = self.pairs[idx]
        img8 = read_geotiff(img_path)
        img = select_rgbnir(img8, self.bands)   # (4,H,W)

        with rasterio.open(lbl_path) as src:
            lbl = src.read(1)
        lbl = (lbl > 0).astype(np.float32)

        if self.normalize:
            img = minmax_per_band(img)

        img_hwc = np.transpose(img, (1,2,0))    # (H,W,4)
        lbl_hwc = lbl[..., None]

        if self.aug:
            transformed = self.aug(image=img_hwc, mask=lbl_hwc)
            img_hwc = transformed["image"]
            lbl_hwc = transformed["mask"]

        img = torch.from_numpy(np.transpose(img_hwc, (2,0,1))).float()
        lbl = torch.from_numpy(lbl_hwc[...,0]).float()
        return img, lbl

train_aug = A.Compose([A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5)])
val_aug   = A.Compose([])

train_ds = LakeTiles(train_pairs, bands=BANDS, aug=None, normalize=True)
val_ds   = LakeTiles(val_pairs,   bands=BANDS, aug=None,   normalize=True)

PIN_MEMORY = False
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          pin_memory=PIN_MEMORY)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          pin_memory=PIN_MEMORY)

len(train_ds), len(val_ds)


(640, 160)

## Loss & Metrics

In [7]:

import torch.nn.functional as F

def dice_loss_with_logits(logits, targets, eps=1e-7):
    probs = torch.sigmoid(logits)
    if probs.dim() == 4:
        probs = probs[:,0]
    intersection = (probs * targets).sum(dim=(1,2))
    union = probs.sum(dim=(1,2)) + targets.sum(dim=(1,2)) + eps
    dice = (2 * intersection + eps) / union
    return 1 - dice.mean()

def bce_dice_loss(logits, targets):
    return F.binary_cross_entropy_with_logits(logits, targets) + dice_loss_with_logits(logits, targets)

def compute_metrics_from_logits(logits, targets):
    with torch.no_grad():
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()
        t = targets.float()
        tp = (preds * t).sum(dim=(1,2))
        tn = ((1-preds) * (1-t)).sum(dim=(1,2))
        fp = (preds * (1-t)).sum(dim=(1,2))
        fn = ((1-preds) * t).sum(dim=(1,2))
        eps = 1e-7
        precision = (tp / (tp + fp + eps)).mean().item()
        recall    = (tp / (tp + fn + eps)).mean().item()
        f1        = (2*precision*recall / (precision + recall + eps))
        iou       = (tp / (tp + fp + fn + eps)).mean().item()
        accuracy  = ((tp + tn) / (tp + tn + fp + fn + eps)).mean().item()
        return dict(iou=iou, precision=precision, recall=recall, f1=f1, accuracy=accuracy)


## Model — Inflate First Conv 3→4

In [ ]:

import segmentation_models_pytorch as smp
import torch
import torch.nn as nn
from torch import optim
import pandas as pd
from datetime import datetime

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def inflate_first_conv_to_8ch(model: nn.Module):
    first_conv = None
    for m in model.encoder.modules():
        if isinstance(m, nn.Conv2d) and m.in_channels == 3:
            first_conv = m
            break
    if first_conv is None:
        return model

    w = first_conv.weight.data
    out_ch, in_ch, kh, kw = w.shape
    new_conv = nn.Conv2d(8, out_ch, kernel_size=first_conv.kernel_size,
                         stride=first_conv.stride, padding=first_conv.padding,
                         dilation=first_conv.dilation, bias=(first_conv.bias is not None))
    with torch.no_grad():
        mean_rgb = w.mean(dim=1, keepdim=True)
        new_w = torch.cat([w, mean_rgb], dim=1)   # append NIR as mean of RGB
        new_conv.weight.copy_(new_w)
        if first_conv.bias is not None and new_conv.bias is not None:
            new_conv.bias.copy_(first_conv.bias.data)

    # replace in encoder
    for name, module in model.encoder.named_modules():
        if module is first_conv:
            parent = model.encoder
            parts = name.split(".")
            for p in parts[:-1]:
                parent = getattr(parent, p)
            setattr(parent, parts[-1], new_conv)
            break
    return model

def make_Segformer(backbone: str, num_classes: int = 1, pretrained=True):
    encoder_weights = "imagenet" if pretrained else None
    model = smp.Segformer(encoder_name=backbone, encoder_weights=encoder_weights,
                     in_channels=8, classes=num_classes)
    if pretrained:
        model = inflate_first_conv_to_8ch(model)
        model.encoder._in_channels = 8
    else:
        model = smp.UnetPlusPlus(encoder_name=backbone, encoder_weights=None,
                         in_channels=8, classes=num_classes)
    return model

def run_one_epoch(model, loader, optimizer=None, use_amp=True):
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0; n_batches = 0
    agg = dict(iou=0.0, precision=0.0, recall=0.0, f1=0.0, accuracy=0.0)

    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        if is_train: optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(imgs)[:,0]
            loss = bce_dice_loss(logits, lbls)

        if is_train:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        total_loss += loss.item(); n_batches += 1
        m = compute_metrics_from_logits(logits.detach(), lbls.detach())
        for k in agg: agg[k] += m[k]

    avg_loss = total_loss / max(1, n_batches)
    for k in agg: agg[k] /= max(1, n_batches)
    torch.cuda.empty_cache()
    return avg_loss, agg

# Stabilize
#torch.backends.cudnn.benchmark = False
#torch.backends.cudnn.deterministic = True


Device: cuda


## Training

In [ ]:

results_csv = OUTPUT_DIR / "logs" / "results.csv"
if not results_csv.exists():
    pd.DataFrame(columns=[
        "timestamp","backbone","epoch","split",
        "loss","iou","precision","recall","f1","accuracy"
    ]).to_csv(results_csv, index=False)

for backbone in BACKBONES:
    print(f"\n=== Training backbone: {backbone} ===")
    model = make_Segformer(backbone, num_classes=1, pretrained=True).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    best_val_iou = -1.0
    best_path = OUTPUT_DIR / "models" / f"{backbone}_best_val_iou.pt"

    for epoch in range(1, EPOCHS+1):
        train_loss, train_metrics = run_one_epoch(model, train_loader, optimizer, use_amp=True)
        val_loss,   val_metrics   = run_one_epoch(model, val_loader, optimizer=None, use_amp=True)

        print(f"[{backbone}] Epoch {epoch}/{EPOCHS} — "
              f"train IoU={train_metrics['iou']:.4f}  val IoU={val_metrics['iou']:.4f}  "
              f"train Loss={train_loss:.4f}  val Loss={val_loss:.4f}")

        ts = datetime.utcnow().isoformat()
        pd.DataFrame([
            dict(timestamp=ts, backbone=backbone, epoch=epoch, split="train", loss=train_loss, **train_metrics),
            dict(timestamp=ts, backbone=backbone, epoch=epoch, split="val",   loss=val_loss,   **val_metrics),
        ]).to_csv(results_csv, mode="a", index=False, header=False)

        if val_metrics["iou"] > best_val_iou:
            best_val_iou = val_metrics["iou"]
            torch.save({
                "backbone": backbone,
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "best_val_iou": best_val_iou,
            }, best_path)

print(f"Training complete. Logs: {results_csv}")
print(f"Models saved under: {OUTPUT_DIR / 'models'}")



=== Training backbone: timm-efficientnet-b0 ===


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

/tmp/ipython-input-67958094.py:56: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipython-input-67958094.py:67: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[timm-efficientnet-b0] Epoch 1/10 — train IoU=0.3946  val IoU=0.4649  train Loss=0.7005  val Loss=0.5253


/tmp/ipython-input-2506835126.py:24: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().isoformat()


[timm-efficientnet-b0] Epoch 2/10 — train IoU=0.5106  val IoU=0.5375  train Loss=0.4545  val Loss=0.4285
[timm-efficientnet-b0] Epoch 3/10 — train IoU=0.5301  val IoU=0.5795  train Loss=0.4316  val Loss=0.3734
[timm-efficientnet-b0] Epoch 4/10 — train IoU=0.5839  val IoU=0.5634  train Loss=0.3708  val Loss=0.4051
[timm-efficientnet-b0] Epoch 5/10 — train IoU=0.5824  val IoU=0.6144  train Loss=0.3664  val Loss=0.3437
[timm-efficientnet-b0] Epoch 6/10 — train IoU=0.6075  val IoU=0.5733  train Loss=0.3399  val Loss=0.3886
[timm-efficientnet-b0] Epoch 7/10 — train IoU=0.6096  val IoU=0.5744  train Loss=0.3340  val Loss=0.3775
[timm-efficientnet-b0] Epoch 8/10 — train IoU=0.6184  val IoU=0.6128  train Loss=0.3271  val Loss=0.3491
[timm-efficientnet-b0] Epoch 9/10 — train IoU=0.6117  val IoU=0.6042  train Loss=0.3325  val Loss=0.3756
[timm-efficientnet-b0] Epoch 10/10 — train IoU=0.6178  val IoU=0.6361  train Loss=0.3279  val Loss=0.3140
Training complete. Logs: /content/drive/My Drive/MS Da